# 04 — Feature Engineering

We create features that could be available when a customer places an order. The model must not see the actual delivery timestamp, the target, review scores, or other information that only becomes available after the order is delivered.

Run `02_clean.ipynb` first.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
source = ROOT / "data/processed/orders_clean.csv"
df = pd.read_csv(source, parse_dates=["order_purchase_timestamp"])
print("Starting shape:", df.shape)

Starting shape: (96470, 24)


## 1. Approximate distance

In [2]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = [np.radians(v) for v in (lat1, lon1, lat2, lon2)]
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon / 2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_km(
    df["seller_lat"], df["seller_lng"],
    df["customer_lat"], df["customer_lng"]
)

## 2. Purchase-time calendar features

In [3]:
purchase = df["order_purchase_timestamp"]
df["purchase_year"] = purchase.dt.year
df["purchase_month"] = purchase.dt.month
df["purchase_day"] = purchase.dt.day
df["purchase_day_of_week"] = purchase.dt.dayofweek
df["purchase_hour"] = purchase.dt.hour
df["is_weekend"] = (purchase.dt.dayofweek >= 5).astype("int8")
df["is_evening"] = purchase.dt.hour.between(18, 23).astype("int8")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_9164\3007558677.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["purchase_day_of_week"] = purchase.dt.dayofweek
C:\Users\Administrator\AppData\Local\Temp\ipykernel_9164\3007558677.py:7: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["is_weekend"] = (purchase.dt.dayofweek >= 5).astype("int8")


## 3. Planned delivery window and order value features

In [4]:
df["estimated_delivery_days"] = (
    pd.to_datetime(df["order_estimated_delivery_date"]) -
    df["order_purchase_timestamp"]
).dt.total_seconds() / 86400

df["freight_share"] = (
    df["total_freight"] / df["total_price"].replace(0, np.nan)
)
df["average_item_price"] = df["total_price"] / df["item_count"].replace(0, np.nan)
df["average_freight_per_item"] = df["total_freight"] / df["item_count"].replace(0, np.nan)

display(df[[
    "distance_km", "estimated_delivery_days", "freight_share",
    "average_item_price", "average_freight_per_item"
]].describe().T)

,count,mean,std,min,25%,50%,75%,max
distance_km,95992.0,600.757241,593.711747,0.000000,185.933322,433.872060,798.548675,8677.911622
estimated_delivery_days,96470.0,23.736343,8.761052,2.008009,18.329905,23.230880,28.407795,155.135463
freight_share,96470.0,0.308386,0.311618,0.000000,0.132017,0.224374,0.380567,21.447059
average_item_price,96470.0,125.231513,189.704435,0.850000,41.900000,79.000000,139.900000,6735.000000
average_freight_per_item,96470.0,20.148509,15.686988,0.000000,13.370000,16.350000,21.160000,409.680000


## 4. Choose model features and prevent leakage

In [5]:
target = "late"

numeric_features = [
    "total_price", "total_freight", "item_count", "payment_value",
    "payment_installments", "distance_km", "estimated_delivery_days",
    "freight_share", "average_item_price", "average_freight_per_item",
    "purchase_month", "purchase_day_of_week", "purchase_hour",
    "is_weekend", "is_evening"
]
categorical_features = [
    "payment_type", "product_category_name", "seller_id",
    "customer_state", "seller_state"
]

feature_columns = numeric_features + categorical_features
model_df = df[feature_columns + [target, "order_purchase_timestamp"]].copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.sort_values("order_purchase_timestamp").reset_index(drop=True)

print("Modeling table shape:", model_df.shape)
print("Late rate:", f"{model_df[target].mean():.2%}")
display(model_df.head())

Modeling table shape: (96470, 22)
Late rate: 8.11%


,total_price,total_freight,item_count,payment_value,payment_installments,distance_km,estimated_delivery_days,freight_share,average_item_price,average_freight_per_item,...,purchase_hour,is_weekend,is_evening,payment_type,product_category_name,seller_id,customer_state,seller_state,late,order_purchase_timestamp
0,134.97,8.49,3,NaN,NaN,565.959812,18.488449,0.062903,44.99,2.83,...,12,0,0,NaN,beleza_saude,ecccfa2bb93b34a3bf033cc5d1dcdc69,SP,PR,1,2016-09-15 12:16:38
1,29.90,15.56,1,45.46,1.0,708.754247,23.593866,0.520401,29.90,15.56,...,9,0,0,boleto,relogios_presentes,522620dcb18a6b31cd7bdf73665113a9,SP,PR,0,2016-10-03 09:44:50
2,21.90,17.19,1,39.09,1.0,915.912412,34.293866,0.784932,21.90,17.19,...,16,0,0,boleto,esporte_lazer,f09b760d23495ac9a7e00d29b769007c,RS,SP,0,2016-10-03 16:56:50
3,36.49,17.24,1,53.73,1.0,818.146504,56.115556,0.472458,36.49,17.24,...,21,0,1,boleto,esporte_lazer,45d33f715e24d15a6ccf5c17b3a23e3c,RS,SP,0,2016-10-03 21:13:36
4,119.90,13.56,1,133.46,6.0,212.014865,50.079132,0.113094,119.90,13.56,...,22,0,1,credit_card,moveis_decoracao,cca3071e3e9bb7d12640c9fbe2301306,SP,SP,0,2016-10-03 22:06:03


## 5. Save engineered dataset

In [6]:
output = ROOT / "data/processed/orders_features.csv"
model_df.to_csv(output, index=False)
print("Saved:", output)

Saved: c:\Users\Administrator\Desktop\olist_notebooks\data\processed\orders_features.csv


## Leakage checklist

The following columns are intentionally excluded from predictors:
- `order_delivered_customer_date`
- `order_delivered_carrier_date`
- `order_estimated_delivery_date` as a raw timestamp
- `delivery_difference_days`
- `late` (target)
- review score and review comments

The **number of estimated delivery days** is allowed because the estimated date is known when the order is placed. It is converted into a duration relative to purchase time.